# AI Studio on Colab (Real Models)

Runs your Next.js app plus a local model gateway in Colab.

Default stack (recommended):
- Chat: `Qwen/Qwen2.5-3B-Instruct` (fast and Colab-friendly)
- Image: `stabilityai/sdxl-turbo`

For higher quality chat, set `CHAT_MODEL` to `Qwen/Qwen2.5-7B-Instruct` if your Colab GPU can handle it.

In [1]:
REPO_URL = "https://github.com/kamalesh4044/colab.git"  # change this
REPO_DIR = "/content/web_ai"
NODE_MAJOR = "20"

CHAT_MODEL = "Qwen/Qwen2.5-3B-Instruct"  # or Qwen/Qwen2.5-7B-Instruct
IMAGE_MODEL = "stabilityai/sdxl-turbo"
HF_TOKEN = ""  # optional, required only for gated/private models

## 1) Install system dependencies

In [2]:
%%bash
set -e
apt-get update -y
apt-get install -y curl git xz-utils
NODE_VERSION=20.20.0
ARCH=linux-x64
cd /usr/local
curl -fsSL "https://nodejs.org/dist/v${NODE_VERSION}/node-v${NODE_VERSION}-${ARCH}.tar.xz" -o node.tar.xz
tar -xJf node.tar.xz
rm node.tar.xz
ln -sfn "/usr/local/node-v${NODE_VERSION}-${ARCH}/bin/node" /usr/local/bin/node
ln -sfn "/usr/local/node-v${NODE_VERSION}-${ARCH}/bin/npm" /usr/local/bin/npm
ln -sfn "/usr/local/node-v${NODE_VERSION}-${ARCH}/bin/npx" /usr/local/bin/npx
node -v
npm -v

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.0 kB]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,378 kB]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:13 http://security.ubuntu.com/ubuntu jammy

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


## 2) Clone and install web app

In [3]:
import os, shutil
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
!git clone "$REPO_URL" "$REPO_DIR"
%cd /content/web_ai
!npm install

Cloning into '/content/web_ai'...
remote: Enumerating objects: 53, done.
remote: Counting objects: 100% (53/53), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 53 (delta 7), reused 53 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (53/53), 96.26 KiB | 12.03 MiB/s, done.
Resolving deltas: 100% (7/7), done.
/content/web_ai
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙
added 201 packages, and audited 202 packages in 18s
⠙
⠙92 packages are looking for funding
⠙  run `npm fund` for details
⠙
1 high severity vulnerability

To address all issues (including breaking changes), run:
  npm audit fix --force

Run `npm audit` for details.
⠹

## 3) Install Python model gateway dependencies

In [4]:
!pip -q install fastapi uvicorn pydantic transformers accelerate sentencepiece safetensors diffusers huggingface_hub

In [5]:
if HF_TOKEN:
    from huggingface_hub import login
    login(HF_TOKEN)

## 4) Create model gateway (`/chat`, `/generate-image`)

In [6]:
%%writefile /content/model_gateway.py
import base64
import io
import os
from typing import List, Optional

import torch
from fastapi import FastAPI
from pydantic import BaseModel
from transformers import AutoModelForCausalLM, AutoTokenizer
from diffusers import StableDiffusionXLPipeline

CHAT_MODEL = os.environ.get("CHAT_MODEL", "Qwen/Qwen2.5-3B-Instruct")
IMAGE_MODEL = os.environ.get("IMAGE_MODEL", "stabilityai/sdxl-turbo")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

print(f"Loading chat model: {CHAT_MODEL}")
tokenizer = AutoTokenizer.from_pretrained(CHAT_MODEL, trust_remote_code=True)
chat_model = AutoModelForCausalLM.from_pretrained(
    CHAT_MODEL,
    trust_remote_code=True,
    torch_dtype=DTYPE,
    device_map="auto" if DEVICE == "cuda" else None,
)
if DEVICE != "cuda":
    chat_model = chat_model.to(DEVICE)
chat_model.eval()

print(f"Loading image model: {IMAGE_MODEL}")
if DEVICE == "cuda":
    image_pipe = StableDiffusionXLPipeline.from_pretrained(
        IMAGE_MODEL,
        torch_dtype=torch.float16,
        variant="fp16",
    ).to("cuda")
else:
    image_pipe = StableDiffusionXLPipeline.from_pretrained(
        IMAGE_MODEL,
        torch_dtype=torch.float32,
    ).to("cpu")

image_pipe.set_progress_bar_config(disable=True)
if DEVICE == "cuda":
    image_pipe.enable_attention_slicing()

app = FastAPI(title="AI Studio Gateway")

class Msg(BaseModel):
    role: str
    content: str

class ChatReq(BaseModel):
    messages: List[Msg]
    model: Optional[str] = None
    max_tokens: int = 512
    temperature: float = 0.7

class ImgReq(BaseModel):
    prompt: str
    steps: int = 4
    model: Optional[str] = None

@app.get("/health")
def health():
    return {"ok": True, "device": DEVICE, "chat_model": CHAT_MODEL, "image_model": IMAGE_MODEL}

@app.post("/chat")
def chat(req: ChatReq):
    messages = [{"role": m.role, "content": m.content} for m in req.messages]
    if hasattr(tokenizer, "apply_chat_template"):
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        prompt = "\n".join([f"{m['role']}: {m['content']}" for m in messages]) + "\nassistant:"

    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(chat_model.device) for k, v in inputs.items()}

    with torch.no_grad():
        output = chat_model.generate(
            **inputs,
            max_new_tokens=min(max(req.max_tokens, 64), 1024),
            do_sample=True,
            temperature=max(req.temperature, 0.1),
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return {"text": text}

@app.post("/generate-image")
def generate_image(req: ImgReq):
    result = image_pipe(
        prompt=req.prompt,
        num_inference_steps=max(1, min(req.steps, 8)),
        guidance_scale=0.0,
    )
    image = result.images[0]
    buf = io.BytesIO()
    image.save(buf, format="PNG")
    b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
    return {"image": f"data:image/png;base64,{b64}"}


Writing /content/model_gateway.py


## 5) Start gateway server

In [18]:
import os, subprocess, time
os.environ["CHAT_MODEL"] = CHAT_MODEL
os.environ["IMAGE_MODEL"] = IMAGE_MODEL
gateway_log = open('/content/gateway.log', 'w')
gateway = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'model_gateway:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd='/content',
    stdout=gateway_log,
    stderr=subprocess.STDOUT
)
print('Gateway PID:', gateway.pid)
time.sleep(10)
!tail -n 80 /content/gateway.log

Gateway PID: 2804


## 6) Wire app env to gateway

In [19]:
%%bash
cd /content/web_ai
cat > .env.local <<EOF
CHAT_ENDPOINT=http://127.0.0.1:8000/chat
CHAT_MODEL=${CHAT_MODEL}
IMAGE_ENDPOINT=http://127.0.0.1:8000/generate-image
IMAGE_MODEL=${IMAGE_MODEL}
# Leave video/transition empty to keep mock fallback
VIDEO_ENDPOINT=
TRANSITION_ENDPOINT=
EOF
cat .env.local

CHAT_ENDPOINT=http://127.0.0.1:8000/chat
CHAT_MODEL=Qwen/Qwen2.5-3B-Instruct
IMAGE_ENDPOINT=http://127.0.0.1:8000/generate-image
IMAGE_MODEL=stabilityai/sdxl-turbo
# Leave video/transition empty to keep mock fallback
VIDEO_ENDPOINT=
TRANSITION_ENDPOINT=


In [33]:
!curl -s -o - -w '\nHTTP %{http_code}\n' http://127.0.0.1:8000/health


{"ok":true,"device":"cuda","chat_model":"Qwen/Qwen2.5-3B-Instruct","image_model":"stabilityai/sdxl-turbo"}
HTTP 200


In [32]:
    !tail -n 120 /content/gateway.log


Loading pipeline components...: 100%|██████████| 7/7 [00:04<00:00,  1.56it/s]
INFO:     Started server process [2804]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


## 7) Start Next.js app

In [34]:
import subprocess, os
os.chdir('/content/web_ai')
next_log = open('/content/next.log', 'w')
next_proc = subprocess.Popen(
    ['npm', 'run', 'dev', '--', '--hostname', '0.0.0.0', '--port', '3000'],
    stdout=next_log,
    stderr=subprocess.STDOUT
)
print('Next.js PID:', next_proc.pid)
!sleep 8; tail -n 80 /content/next.log

Next.js PID: 3468

> ai-studio-mock@1.0.0 dev
> next dev --hostname 0.0.0.0 --port 3000

  ▲ Next.js 14.2.35
  - Local:        http://localhost:3000
  - Network:      http://0.0.0.0:3000
  - Environments: .env.local

 ✓ Starting...
 ✓ Ready in 2.4s


## 8) Expose public URL

In [10]:
!wget -q -O /content/cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i /content/cloudflared.deb
!cloudflared tunnel --url http://localhost:3000 --no-autoupdate

Selecting previously unselected package cloudflared.
(Reading database ... 121852 files and directories currently installed.)
Preparing to unpack /content/cloudflared.deb ...
Unpacking cloudflared (2026.2.0) ...
Setting up cloudflared (2026.2.0) ...
Processing triggers for man-db (2.10.2-1) ...
2026-02-24T14:34:44Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-02-24T14:34:44Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-02-24T14:34:48Z INF 